# Teaching a computer to see a galaxy more clearly

### A ten-minute look at how AI actually gets used in astronomy

Every image a telescope takes is blurrier than the sky really is. The
atmosphere smears it, the optics smear it, and the detector only has so many
pixels. Astronomers have fought this for four hundred years.

What follows is a small neural network that was shown a few hundred galaxies
and taught itself to undo some of that damage. It is running on this laptop,
right now, on images it has never seen.

Nothing here is downloaded live and nothing is trained live - that part
happened yesterday, and took about ten minutes.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt

from galaxy_sr import data, viz
from galaxy_sr.model import load_trained

model, info = load_trained("weights/galaxy_sr.pt")
names = [f.stem for f in sorted(Path("data/demo").glob("*.png"))]
galaxies = dict(zip(names, data.load_images("data/demo")))

print(f"Model loaded: {model.n_params:,} adjustable numbers")
print(f"Trained on {info['n_train_images']} galaxies in "
      f"{info['minutes']:.1f} minutes")
print(f"Galaxies available today: {', '.join(list(galaxies)[:8])} ...")

---
## 1. Here is the truth

A real galaxy, photographed by a large survey telescope. Hold this image in
your head - it is the answer key.

In [ ]:
NAME = list(galaxies)[0]          # <-- change this to pick a different galaxy
truth = galaxies[NAME]

plt.figure(figsize=(7.5, 7.5))
plt.imshow(truth); plt.axis("off")
plt.title(NAME.replace("_", " "), fontsize=18)
plt.show()

---
## 2. Now let's ruin it

I am going to simulate a much smaller, cheaper telescope: blur the image the
way the atmosphere does, throw away 15 out of every 16 pixels, and add the
electrical noise a real detector adds.

This is not a trick - it is the standard way this problem is set up. We
*have* to damage a good image on purpose, because that is the only way to
get a matched pair of "bad picture" and "correct answer" to learn from.

In [ ]:
res = viz.run_all(model, truth)

print("Original :", tuple(res["truth"].shape[-2:]), "pixels")
print("Degraded :", tuple(res["low"].shape[-2:]), "pixels  "
      f"({100 * res['low'][0,0].numel() / res['truth'][0,0].numel():.0f}% "
      "of the information left)")

plt.figure(figsize=(7.5, 7.5))
plt.imshow(viz.to_numpy(res["low"]), interpolation="nearest"); plt.axis("off")
plt.title("What the small telescope sees", fontsize=18)
plt.show()

**Worth asking the room here:** how much of that do you think is
recoverable? Most people say "almost none - the information is gone."

---
## 3. The reveal

Four versions of the same galaxy. The second panel is the ordinary way to
enlarge an image - the same thing your photo app does when you zoom in. The
third is the neural network.

In [ ]:
t0 = time.time()
res = viz.run_all(model, truth)
elapsed = time.time() - t0

viz.compare(res, save=f"figures/{NAME}_compare.png")
plt.show()
print(f"The AI took {elapsed:.2f} seconds.")

---
## 4. Closer

Same square of sky, blown up, in all four versions.

In [ ]:
X, Y, SIZE = 190, 190, 130     # <-- move the zoom box around
viz.zoom_in(res, X, Y, SIZE, save=f"figures/{NAME}_zoom.png")
plt.show()

---
## 5. The part that matters most: where it is *wrong*

This is the slide I would want a student to understand before any of the
rest of it.

The network does not "reveal" hidden detail. It has learned what galaxies
tend to look like, and it uses that expectation to make an educated guess.
Usually the guess is very good. Sometimes it invents a feature that is not
there - a plausible smudge where the real sky had nothing.

That is why an astronomer never publishes an AI-sharpened image as evidence
of a discovery. We use it to decide where to point the expensive telescope
next. **The AI narrows the search; the human confirms the finding.**

Below, brighter means further from the truth.

In [ ]:
viz.where_it_erred(res, save=f"figures/{NAME}_errors.png")
plt.show()

---
## 6. Does it actually work, or did I pick a lucky picture?

Scored on galaxies the model never saw during training. Higher is closer to
the truth; every 6 dB is roughly a halving of the error.

In [ ]:
viz.scoreboard(info, save="figures/scoreboard.png")
plt.show()
print(f"Improvement: +{info['psnr_model'] - info['psnr_bicubic']:.2f} dB "
      f"over the standard method, measured on {info['n_val_images']} "
      "held-out galaxies.")

---
## 7. Pick one

Someone call out a name from the list and I will run it.

In [ ]:
print(list(galaxies))

In [ ]:
PICK = list(galaxies)[1]       # <-- type the name the room chose

res2 = viz.run_all(model, galaxies[PICK])
viz.compare(res2, save=f"figures/{PICK}_compare.png")
plt.show()

---
## 8. Why any of this matters at scale

One galaxy is a party trick. The reason observatories care is the arithmetic
below.

In [ ]:
t0 = time.time()
for name, im in galaxies.items():
    viz.run_all(model, im)
per_galaxy = (time.time() - t0) / len(galaxies)

survey = 100_000_000     # roughly what a modern sky survey will catalogue
print(f"{per_galaxy:.3f} seconds per galaxy on this machine")
print(f"A person inspecting these by eye at 30 seconds each would need "
      f"{survey * 30 / 3600 / 24 / 365:,.0f} years for a full survey.")
print("That is the actual reason the field adopted this. Not novelty - "
      "arithmetic.")

---
# What a student would learn building this

Not one of these is astronomy. Every one of them is a hiring requirement
somewhere.

| What they did | What it is called in industry |
|---|---|
| Fetched images from a public scientific archive | working with APIs and real, messy data |
| Damaged good images on purpose to make training pairs | dataset design - the part most people skip |
| Chose a network small enough to train in ten minutes | working inside a real compute budget |
| Held back images to grade the model honestly | validation; not fooling yourself |
| Compared against the boring non-AI method | benchmarking - is the AI even worth it? |
| Found where the model invents detail | model evaluation, failure analysis, AI ethics |
| Knew when *not* to trust the output | the judgment that separates a user from an operator |

The last two rows are the ones I care about. A student who has watched a
model confidently produce a beautiful wrong answer, in a case where they
happen to hold the answer key, has learned something about AI that no amount
of warning them can teach.

That is the argument for letting this into the classroom rather than leaving
it at the door. They are going to use these tools regardless. The only
question is whether they learn to check them here, with a galaxy, where
being wrong is free - or somewhere later, where it isn't.

---

*Everything in this notebook is open: the images are public data from the
Sloan Digital Sky Survey, the code is a few hundred lines, and it runs on any
machine with a graphics card. Happy to hand the whole thing to anyone who
wants to try it with a class.*